# 02 — Process raw RTT extracts into one tidy file (96 months)

Turns all monthly raw CSVs on disk (96 on the A100) (~100 MB each, 121 columns) into **one small tidy file**:
`data/processed/rtt_long.csv` — one row per `(month, trust, part, specialty)`.

Steps per raw file:
1. keep only the useful columns (ids + 105 week bands + totals)
2. drop specialty-total rows (`C_999`) — they would double-count
3. keep **NHS trusts only** (3-char org code starting with `R`) → removes private providers
4. keep only the **35 network trusts** from `config/nhs_network.json`
5. compress the 105 week bands → `weeks_0_18` / `weeks_18_52` / `weeks_52_plus` (18w = NHS standard, 52w = long-wait threshold)
6. group by `(month, trust, part, specialty)` and **sum across commissioners** (raw rows are split by who pays)

**Data rule discovered by reconciliation:** counts live in `Total All` — the raw `Total` column is empty for Part_2/2A/3.

In [1]:
import glob
import json
import os
import pandas as pd

RAW_DIR = "../data/nhs_rtt_referral_to_treatment"
OUT_CSV = "../data/processed/rtt_long.csv"

# identity columns we keep from the 121 in each raw file
ID_COLS = [
    "Period",
    "Provider Org Code",
    "Provider Org Name",
    "RTT Part Type",
    "Treatment Function Code",
    "Treatment Function Name",
]

# 105 week-band columns: "Gt 00 To 01 Weeks SUM 1" ... "Gt 103 To 104 Weeks SUM 1", "Gt 104 Weeks SUM 1"
BAND_COLS = [f"Gt {i:02d} To {i + 1:02d} Weeks SUM 1" for i in range(104)] + ["Gt 104 Weeks SUM 1"]

BANDS_0_18 = BAND_COLS[:18]      # waited <= 18 weeks (inside the NHS standard)
BANDS_18_52 = BAND_COLS[18:52]   # > 18 and <= 52 weeks (breach)
BANDS_52_PLUS = BAND_COLS[52:]   # > 52 weeks (long waiters)

TOTAL_COLS = ["Total", "Total All"]   # counts live in "Total All"
USE_COLS = ID_COLS + BAND_COLS + TOTAL_COLS
print(f"columns kept per raw file: {len(USE_COLS)} of 121")

columns kept per raw file: 113 of 121


In [2]:
# the 35 federated clients, straight from the config (no hard-coded list)
with open("../config/nhs_network.json") as f:
    network = json.load(f)
NETWORK_CODES = sorted(t["code"] for r in network["regions"] for t in r["trusts"])
print(f"network trusts from config: {len(NETWORK_CODES)}")

network trusts from config: 35


In [3]:
# process every monthly file (takes a few minutes: ~5 GB of CSV)
files = sorted(glob.glob(os.path.join(RAW_DIR, "rtt_*.csv")))
print(f"raw monthly files: {len(files)}")

monthly = []
for path in files:
    month = os.path.basename(path)[4:11]  # rtt_2021-04.csv -> 2021-04
    df = pd.read_csv(path, usecols=USE_COLS, dtype={c: str for c in ID_COLS})

    n_providers_raw = df["Provider Org Code"].nunique()

    # drop specialty-total rows, keep real specialties only
    df = df[df["Treatment Function Code"] != "C_999"]

    # NHS trusts only: 3-char org code starting with R  -> removes private providers
    df = df[df["Provider Org Code"].str.match(r"^R[A-Z0-9]{2}$")]
    n_providers_nhs = df["Provider Org Code"].nunique()

    # network trusts only (the 35 federated clients)
    df = df[df["Provider Org Code"].isin(NETWORK_CODES)]

    # week bands + totals to numeric (empty strings -> 0)
    df[BAND_COLS + TOTAL_COLS] = df[BAND_COLS + TOTAL_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

    # compress 105 bands into 3 operational aggregates (all at once, no fragmentation)
    df = df.assign(
        weeks_0_18=df[BANDS_0_18].sum(axis=1),
        weeks_18_52=df[BANDS_18_52].sum(axis=1),
        weeks_52_plus=df[BANDS_52_PLUS].sum(axis=1),
    )

    # aggregate across commissioners -> one row per (trust x part x specialty)
    out = (
        df.groupby(["Provider Org Code", "Provider Org Name",
                    "RTT Part Type", "Treatment Function Code", "Treatment Function Name"],
                   as_index=False)[["weeks_0_18", "weeks_18_52", "weeks_52_plus", "Total", "Total All"]]
        .sum()
    )
    out.insert(0, "month", month)
    out.columns = ["month", "provider_code", "provider_name", "part",
                   "specialty_code", "specialty_name",
                   "weeks_0_18", "weeks_18_52", "weeks_52_plus", "total", "total_all"]
    monthly.append(out)

    print(f"{month}: providers {n_providers_raw:>3} -> NHS {n_providers_nhs:>3} "
          f"-> network {out['provider_code'].nunique():>2} | out rows {len(out):>5}")

raw monthly files: 60


C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

2021-04: providers 431 -> NHS 170 -> network 35 | out rows  3479


C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

2021-05: providers 428 -> NHS 170 -> network 35 | out rows  3487


C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

... (111 outputs left out)

C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

2026-02: providers 515 -> NHS 149 -> network 35 | out rows  3498


2026-03: providers 533 -> NHS 149 -> network 35 | out rows  3494


C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(
C:\Users\heave\AppData\Local\Temp\ipykernel_8600\530625826.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

In [4]:
# combine all months and save the single tidy file
long = pd.concat(monthly, ignore_index=True)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
long.to_csv(OUT_CSV, index=False)
print(f"saved {OUT_CSV}: {len(long)} rows x {long.shape[1]} columns")
long.head()

saved ../data/processed/rtt_long.csv: 210328 rows x 11 columns


,month,provider_code,provider_name,part,specialty_code,specialty_name,weeks_0_18,weeks_18_52,weeks_52_plus,total,total_all
0,2021-04,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Part_1A,C_100,General Surgery Service,328.0,139.0,12.0,479.0,479
1,2021-04,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Part_1A,C_101,Urology Service,187.0,85.0,0.0,272.0,272
2,2021-04,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Part_1A,C_110,Trauma and Orthopaedic Service,47.0,99.0,63.0,209.0,209
3,2021-04,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Part_1A,C_120,Ear Nose and Throat Service,94.0,32.0,0.0,126.0,126
4,2021-04,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Part_1A,C_130,Ophthalmology Service,394.0,314.0,1.0,709.0,709


In [5]:
# sanity checks
print(f"months: {long['month'].nunique()} ({long['month'].min()} .. {long['month'].max()})")
print(f"trusts: {long['provider_code'].nunique()}")
print(f"parts:  {sorted(long['part'].unique())}")
print(f"specialties: {long['specialty_code'].nunique()}")

# completeness is asserted on the FROZEN window only (pre-registered); earlier months
# (2018-2021) are extra history and MAY be partial for some trusts
WINDOW = ("2021-04", "2026-03")
core = long[long["month"].between(*WINDOW)]
presence = core.groupby("provider_code")["month"].nunique()
assert (presence == core["month"].nunique()).all(), "some trusts missing months in the frozen window!"
print(f"all 35 trusts present in all {core['month'].nunique()} frozen-window months - OK")
first = long.groupby("provider_code")["month"].min()
print("earliest month per trust (history coverage):")
print(first.value_counts().sort_index().to_string())

# reconciliation per part. Raw extract populates totals differently per part:
#   Part_1A/1B: bands + Total          Part_2/2A: bands + Total All (Total empty)
#   Part_3: Total All only (no bands)  -> downstream counts must use total_all
band_sum = long["weeks_0_18"] + long["weeks_18_52"] + long["weeks_52_plus"]
completed = long["part"].isin(["Part_1A", "Part_1B"])
incomplete = long["part"].isin(["Part_2", "Part_2A"])
part3 = long["part"] == "Part_3"
print(f"Part_1A/1B band-sum vs total mismatches:     {((band_sum - long['total']).abs() > 0.5)[completed].sum()}")
print(f"Part_2/2A  band-sum vs total_all mismatches: {((band_sum - long['total_all']).abs() > 0.5)[incomplete].sum()}")
print(f"Part_3     rows with zero total_all:         {(long['total_all'] == 0)[part3].sum()}")

months: 60 (2021-04 .. 2026-03)
trusts: 35
parts:  ['Part_1A', 'Part_1B', 'Part_2', 'Part_2A', 'Part_3']
specialties: 23
all 35 trusts present in all 60 frozen-window months - OK
earliest month per trust (history coverage):
month
2021-04    35
Part_1A/1B band-sum vs total mismatches:     0
Part_2/2A  band-sum vs total_all mismatches: 0
Part_3     rows with zero total_all:         0
